# 06 · Try it yourself: every hands-on exercise

Run the agent's real code yourself, in three chapters. Do each part right after its notebook: chapter 1 after 02, chapter 2 after 03, chapter 3 after 05. **No API key and no payment:** every LLM answer used here was saved from a real LLM and is replayed for $0.

- **On your laptop:** `uv run --group notebooks jupyter lab`, open this notebook, then run the cells one by one (`Shift + Enter`).
- **No setup?** It also runs in Google Colab, in your browser:
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AnkitSurana/agent-on-a-budget/blob/main/notebooks/06_try_it_yourself.ipynb)

In [1]:
# Run this first. In Colab it downloads the project (a few seconds); on your laptop it just finds it.
import os, subprocess, sys

try:
    import google.colab  # only exists in Google Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    repo = os.path.abspath("agent-on-a-budget")
    if not os.path.exists(repo):
        subprocess.run(["git", "clone", "-q", "https://github.com/AnkitSurana/agent-on-a-budget.git", repo], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "python-dotenv", "openai"], check=True)
    os.chdir(repo)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")                      # the project folder
sys.path.insert(0, os.getcwd())
from budget_agent import config
if config.other_python():               # for example, Anaconda's Jupyter instead of the project's
    print(f"⚠️  This is not the project's Python ({config.other_python()}). Close it and run: uv run --group notebooks jupyter lab")
print("Ready: the project is loaded")

Ready: the project is loaded


## Chapter 1: build your first agent

### Exercise 1.1: the tools and the guardrails

These are the exact functions the LLM asks for. Here **you** play the LLM's part. Watch which requests our Python code refuses.

In [2]:
from budget_agent.tools import Shop

shop = Shop()                                           # our pretend NovaMart database
print(shop.lookup_order("NM10009"))                     # a plain lookup
print(shop.start_refund("NM10002", "arrived broken"))   # delivered, under ₹10,000: allowed
print(shop.start_refund("NM10018", "don't like it"))    # ₹12,999: a human must approve
print(shop.cancel_order("NM10009"))                     # already shipped: refused
print(shop.cancel_order("NM10007"))                     # still processing: allowed
shop.actions                                            # everything that changed

{'order_id': 'NM10009', 'customer_name': 'Rohan Das', 'city': 'Bengaluru', 'product': 'Running Shoes', 'quantity': 1, 'amount_inr': 3299, 'order_date': '2026-09-12', 'status': 'Shipped', 'delivery_date': '2026-10-08'}
{'ok': True, 'message': 'Refund of ₹5,999 started. It arrives in 5-7 working days.'}
{'ok': False, 'why': 'Refunds above ₹10,000 need a human. A human agent has been asked to approve it.'}
{'ok': False, 'why': "Order is 'Shipped'. Only 'Processing' orders can be cancelled."}
{'ok': True, 'message': 'Order cancelled. No fee was charged.'}


[{'action': 'start_refund', 'order_id': 'NM10002', 'reason': 'arrived broken'},
 {'action': 'escalate_to_human',
  'ticket': 'T0002',
  'reason': "Refund over ₹10,000 for NM10018: don't like it",
  'priority': 'high'},
 {'action': 'cancel_order', 'order_id': 'NM10007'}]

The ₹12,999 refund was refused and a ticket was opened for a human. That rule lives in Python, so the LLM can't talk its way past it.

### Exercise 1.2: what one LLM call costs

1,000 tokens in and 200 tokens out, with a small model and a big one:

In [3]:
from budget_agent.llm import cost_usd

print(f"gpt-5.4-mini:    ${cost_usd('gpt-5.4-mini', 1000, 200):.5f}")
print(f"claude-opus-5-5: ${cost_usd('claude-opus-5-5', 1000, 200):.5f}")
print(f"llama3.2:3b:     ${cost_usd('llama3.2:3b', 1000, 200):.5f}  (runs on your laptop)")

gpt-5.4-mini:    $0.00165
claude-opus-5-5: $0.00800
llama3.2:3b:     $0.00000  (runs on your laptop)


Tiny for one call. Now multiply by 1 to 4 calls per message and thousands of messages a day.

### Exercise 1.3: the senior expert at work

The LLM with tools runs the full agent loop: it asks for `lookup_order`, our code runs it, and the LLM writes the answer. (This answer was saved from a real run, so it's replayed for $0.)

In [4]:
from budget_agent.agent import SupportAgent

expert = SupportAgent(use_tools=True)
result = expert.run("where is my order NM10009?")
for name, tool_input, output in result.tool_calls:
    print(f"🔧 {name}({tool_input})")
print(f"💬 {result.reply}")
print(f"   {len(result.calls)} LLM calls · ${result.cost_usd:.4f} when first made · replayed now: {result.replayed}")

🔧 lookup_order({'order_id': 'NM10009'})
💬 Your order NM10009 is shipped and expected to be delivered on 08 Oct 2026. If you want, I can also help with anything else about the order.
   2 LLM calls · $0.0011 when first made · replayed now: True


## Chapter 2: the LLM teaches a small ML model

### Exercise 2.1: ask the receptionist

The receptionist is a tiny ML model trained on 1,080 messages the LLM labelled. It guesses the intent and says **how sure** it is. The first run trains it (a few seconds).

Change the messages, or add your own.

In [5]:
from budget_agent import classifier

model = classifier.load_or_train()
for text in ["how do i reset my pasword",
             "where is my parcel NM10005",
             "mera refund kab aayega",
             "i want to talk to a human"]:
    intent, sure = classifier.predict(model, text)
    print(f"{sure:4.0%} sure  →  {intent:24} {text}")

 90% sure  →  recover_password         how do i reset my pasword
 65% sure  →  track_order              where is my parcel NM10005
 30% sure  →  check_refund_policy      mera refund kab aayega
 83% sure  →  contact_human_agent      i want to talk to a human


At 80% or more it answers by itself; below that it passes the message on. Notice the Hinglish one: it knows it doesn't know. (One exception comes in chapter 3: a request that changes something, like "talk to a human", always goes to the expert, however sure the receptionist is.)

### Exercise 2.2: turn the `C` dial

`C` controls how much the model holds back. Same data, three settings. (Takes a few seconds.)

In [6]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from budget_agent import config, data

messages = data.load_messages()
train = messages[messages.split == "train"].merge(pd.read_csv(config.LLM_LABELS_CSV), on="id")  # the LLM's labels
test = messages[messages.split == "test"]
inbox = pd.concat([test[["text", "true_intent"]], data.load_messy()])                        # clean + messy

rows = []
for C in [1, 20, 100]:
    student = make_pipeline(TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True, min_df=2),
                            LogisticRegression(C=C, max_iter=2000)).fit(train.text, train.llm_intent)
    probs = student.predict_proba(inbox.text)
    sure = probs.max(axis=1) >= 0.80
    right = student.classes_[probs.argmax(axis=1)] == inbox.true_intent
    rows.append({"C": C,
                 "right on clean messages": f"{(student.predict(test.text) == test.true_intent).mean():.1%}",
                 "average confidence": f"{student.predict_proba(test.text).max(axis=1).mean():.0%}",
                 "answers at the 80% line": f"{sure.mean():.0%}",
                 "right when it answers": f"{right[sure].mean():.1%}"})
pd.DataFrame(rows)

,C,right on clean messages,average confidence,answers at the 80% line,right when it answers
0,1,92.0%,47%,2%,100.0%
1,20,92.4%,86%,72%,98.3%
2,100,91.9%,92%,83%,96.5%


Same accuracy, very different behaviour: at `C=1` it is too careful to answer almost anything; at `C=100` it answers more but is wrong more often. The project uses `C=20`.

## Chapter 3: the whole support office

### Exercise 3.1: who answers, and the dial

The router picks the cheapest helper from the receptionist's intent and confidence. No LLM is called for this decision.

In [7]:
from budget_agent import config
from budget_agent.router import Router

office = Router()
config.ML_CONFIDENCE = 0.80             # the project's setting
for text in ["track my order NM10005", "can i pay with upi", "cancel my order NM10007"]:
    helper, intent, sure = office.choose_layer(text)
    print(f"{config.HELPERS[helper]:32} {intent:16} {sure:4.0%}  {text}")

config.ML_CONFIDENCE = 0.95             # a stricter dial
helper, intent, sure = office.choose_layer("track my order NM10005")
print(f"\nWith the dial at 95%: {config.HELPERS[helper]} takes 'track my order NM10005'")
config.ML_CONFIDENCE = 0.80             # back to normal

Receptionist (ML model)          track_order       89%  track my order NM10005
Senior expert (cloud LLM agent)  payment_issue     58%  can i pay with upi
Senior expert (cloud LLM agent)  cancel_order      94%  cancel my order NM10007

With the dial at 95%: Senior expert (cloud LLM agent) takes 'track my order NM10005'


- "can i pay with upi" was misread as `payment_issue`, an action, so it went safely to the expert.
- "cancel my order" is 94% sure but changes an order, so the expert takes it anyway.
- A stricter dial is safer, but sends more messages to the paid expert.

### Exercise 3.2: talk to the whole office

In [8]:
office.fresh_shop()
for text in ["track my order NM10005", "how do i reset my pasword", "what payment methods do you accept"]:
    r = office.handle(text)
    print(f"📨 {text}\n   {config.HELPERS[r.layer]} · {r.seconds:.2f}s · ${r.cost_usd:.4f}\n   💬 {r.reply}\n")

📨 track my order NM10005
   Receptionist (ML model) · 0.00s · $0.0000
   💬 Your order NM10005 (Smart Watch) is Delivered. Delivery date: 2026-10-02.

📨 how do i reset my pasword
   Receptionist (ML model) · 0.00s · $0.0000
   💬 Click Sign in, then 'Forgot password'. We will email you a reset link.

📨 what payment methods do you accept
   Receptionist (ML model) · 0.00s · $0.0000
   💬 We accept UPI, credit and debit cards, net banking, and cash on delivery for orders up to ₹5,000.



All three were answered by the receptionist, instantly and for $0.

## Optional: ask your own questions

New questions need an LLM. Put your own key in the cell below (Gemini has a free tier: [aistudio.google.com/apikey](https://aistudio.google.com/apikey)), remove the `#` signs and run it. **Never share your screen with a key on it.**

In [9]:
# Optional: remove the # at the start of each line below, put in your own key, then run this cell.
# import os
# os.environ["LLM_PROVIDER"] = "gemini"        # or openai, anthropic
# os.environ["LLM_API_KEY"] = "AIza..."        # your key
# r = office.handle("can I return a gift I received?")
# print(config.HELPERS[r.layer], "·", f"${r.cost_usd:.4f}", "\n", r.reply)

## What you just did

- **Chapter 1:** called the agent's tools yourself, saw Python refuse a ₹12,999 refund, and ran the full agent loop.
- **Chapter 2:** asked the receptionist, and saw how one setting (`C`) changes how sure it is.
- **Chapter 3:** saw the router pick the cheapest helper, and how the confidence dial trades cost for safety.

The rest of the story is in notebooks 00 to 05.